In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

**Dataset selection**

The selected dataset is UNSW-NB15, a cybersecurity dataset used for network intrusion detection. It contains network traffic records labeled as normal or attack traffic. For this first delivery, the binary label column is used as the target variable.

In [2]:
train = pd.read_csv("/content/UNSW_NB15_training-set.csv")

test = pd.read_csv("/content/UNSW_NB15_testing-set.csv")

**train/test split**

The dataset already includes official training and testing files. Therefore, no additional random split was necessary.

In [5]:
train.tail()
train.info()

print('')

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82332 entries, 0 to 82331
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 82332 non-null  int64  
 1   dur                82332 non-null  float64
 2   proto              82332 non-null  object 
 3   service            82332 non-null  object 
 4   state              82332 non-null  object 
 5   spkts              82332 non-null  int64  
 6   dpkts              82332 non-null  int64  
 7   sbytes             82332 non-null  int64  
 8   dbytes             82332 non-null  int64  
 9   rate               82332 non-null  float64
 10  sttl               82332 non-null  int64  
 11  dttl               82332 non-null  int64  
 12  sload              82332 non-null  float64
 13  dload              82332 non-null  float64
 14  sloss              82332 non-null  int64  
 15  dloss              82332 non-null  int64  
 16  sinpkt             823

In [6]:
train.count()

,0
id,82332
dur,82332
proto,82332
service,82332
state,82332
spkts,82332
dpkts,82332
sbytes,82332
dbytes,82332
rate,82332


In [7]:
print('Train columns with null values:',train.isnull().sum(), sep = '\n')
print("")


print('Test columns with null values:', test.isnull().sum(),sep = '\n')
print("")

Train columns with null values:
id                   0
dur                  0
proto                0
service              0
state                0
spkts                0
dpkts                0
sbytes               0
dbytes               0
rate                 0
sttl                 0
dttl                 0
sload                0
dload                0
sloss                0
dloss                0
sinpkt               0
dinpkt               0
sjit                 0
djit                 0
swin                 0
stcpb                0
dtcpb                0
dwin                 0
tcprtt               0
synack               0
ackdat               0
smean                0
dmean                0
trans_depth          0
response_body_len    0
ct_srv_src           0
ct_state_ttl         0
ct_dst_ltm           0
ct_src_dport_ltm     0
ct_dst_sport_ltm     0
ct_dst_src_ltm       0
is_ftp_login         0
ct_ftp_cmd           0
ct_flw_http_mthd     0
ct_src_ltm           0
ct_srv_dst           0
is

In [8]:
# labels column indicates 1 = attack traffic, 0 = normal traffic

print(f"Train label percentage:\n{train['label'].value_counts(normalize=True)}")
print(f"Test label percentage:\n{test['label'].value_counts(normalize=True)}")

Train label percentage:
label
1    0.5506
0    0.4494
Name: proportion, dtype: float64
Test label percentage:
label
1    0.680622
0    0.319378
Name: proportion, dtype: float64


In [9]:
train.dtypes

,0
id,int64
dur,float64
proto,object
service,object
state,object
spkts,int64
dpkts,int64
sbytes,int64
dbytes,int64
rate,float64


**Preprocessing**

The columns id, label, and attack_cat were removed from the feature matrix. The id column is only an identifier, label is the target variable, and attack_cat would leak information about whether the record is normal or an attack.

In [10]:
# remove leakage columns
excluded_cols = ["id", "label", "attack_cat"]

train_features = train.drop(columns=excluded_cols)
test_features = test.drop(columns=excluded_cols)

categorical_cols = train_features.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = train_features.select_dtypes(exclude=["object"]).columns.tolist()

train_labels = train["label"]
test_labels = test["label"]

**Encoding**

Categorical columns such as proto, service, and state were converted using one-hot encoding so they could be processed by the neural network.

In [14]:
# one-hot enconde categorical columns
train_features_encoded = pd.get_dummies(train_features, columns=categorical_cols)
test_features_encoded = pd.get_dummies(test_features, columns=categorical_cols)

train_features_encoded, test_features_encoded = train_features_encoded.align(
    test_features_encoded,
    join="left",
    axis=1,
    fill_value=0
)

In [12]:
print(train_features_encoded.shape)
print(test_features_encoded.shape)

print(train_features_encoded.head())

train_features_encoded = train_features_encoded.astype(float)
test_features_encoded = test_features_encoded.astype(float)

(82332, 190)
(175341, 190)
        dur  spkts  dpkts  sbytes  dbytes         rate  sttl  dttl  \
0  0.000011      2      0     496       0   90909.0902   254     0   
1  0.000008      2      0    1762       0  125000.0003   254     0   
2  0.000005      2      0    1068       0  200000.0051   254     0   
3  0.000006      2      0     900       0  166666.6608   254     0   
4  0.000010      2      0    2126       0  100000.0025   254     0   

         sload  dload  ...  service_snmp  service_ssh  service_ssl  state_ACC  \
0  180363632.0    0.0  ...         False        False        False      False   
1  881000000.0    0.0  ...         False        False        False      False   
2  854400000.0    0.0  ...         False        False        False      False   
3  600000000.0    0.0  ...         False        False        False      False   
4  850400000.0    0.0  ...         False        False        False      False   

   state_CLO  state_CON  state_FIN  state_INT  state_REQ  state_R

**Scaling**

Numerical features were standardized using sklearn's StandardScaler.

In [13]:
scaler = StandardScaler()

train_features_scaled = scaler.fit_transform(train_features_encoded)
test_features_scaled = scaler.transform(test_features_encoded)

**Basic Neural Network**

A basic example of how a neural network implementation would look applied to this dataset. (Not based on the paper yet)

In [ ]:
input_dim = train_features_scaled.shape[1]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation="relu", input_shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.fit(
    train_features_scaled,
    train_labels,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9587 - loss: 0.1127 - val_accuracy: 0.4093 - val_loss: 0.9516
Epoch 2/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9685 - loss: 0.0734 - val_accuracy: 0.4933 - val_loss: 0.9313
Epoch 3/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9713 - loss: 0.0656 - val_accuracy: 0.5810 - val_loss: 0.9114
Epoch 4/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9742 - loss: 0.0583 - val_accuracy: 0.7030 - val_loss: 0.7938
Epoch 5/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9764 - loss: 0.0557 - val_accuracy: 0.6758 - val_loss: 0.8966
Epoch 6/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9778 - loss: 0.0529 - val_accuracy: 0.6927 - val_loss: 0.8885
Epoch 7/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9785 - loss: 0.0514 - val_accuracy: 0.5844 - val_loss: 1.2225
Epoch 8/10
1030/1030 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9790 - loss: 0.0502 - 

In [ ]:
test_loss, test_accuracy = model.evaluate(test_features_scaled, test_labels)

print("Test accuracy:", test_accuracy)

5480/5480 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9177 - loss: 0.2430
Test accuracy: 0.9176861047744751


In [ ]:
predictions = model.predict(test_features_scaled)

5480/5480 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step


In [ ]:
label_pred = (predictions >= 0.5).astype(int)